# LSTM MODEL TRAINING

**Notebook 05**: Train LSTM models for carbon emission forecasting

**Optimized for M3 MacBook Air (16GB RAM)**
- Small model architecture
- Efficient batch processing
- CPU-friendly training

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import torch
import torch.nn as nn
import joblib
import json
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Check if MPS (Apple Silicon GPU) available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print("Libraries imported successfully")

Using device: mps
Libraries imported successfully


## Load Data

In [2]:
train = pd.read_csv('../data/processed/train_data.csv')
test = pd.read_csv('../data/processed/test_data.csv')

print(f"Train: {len(train)} records")
print(f"Test: {len(test)} records")

Train: 53391 records
Test: 6510 records


## Select Key Series

Training fewer series to keep memory usage low

In [3]:
# Top 5 states only (to save memory and time)
top_states = test[test['fuel-name'] == 'All Fuels'].groupby('state-name')['value'].sum().nlargest(5).index.tolist()

train_subset = train[
    (train['state-name'].isin(top_states)) &
    (train['fuel-name'] == 'All Fuels')
].copy()

test_subset = test[
    (test['state-name'].isin(top_states)) &
    (test['fuel-name'] == 'All Fuels')
].copy()

print(f"Selected {len(top_states)} states for LSTM training")
print(f"States: {', '.join(top_states)}")

Selected 5 states for LSTM training
States: United States, Texas, California, Florida, Pennsylvania


## Define Lightweight LSTM Model

In [4]:
class LightweightLSTM(nn.Module):
    """
    Small LSTM optimized for M3 Mac
    - Single LSTM layer (32 units)
    - Dropout for regularization
    - Linear output layer
    """
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, dropout=0.2):
        super(LightweightLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # LSTM layer
        lstm_out, _ = self.lstm(x)
        
        # Take last output
        last_out = lstm_out[:, -1, :]
        
        # Fully connected layer
        output = self.fc(last_out)
        
        return output

print("LSTM model defined")

LSTM model defined


## Data Preparation Functions

In [5]:
def create_sequences(data, seq_length=5):
    """
    Create sequences for LSTM
    seq_length: how many years to look back
    """
    X, y = [], []
    
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    
    return np.array(X), np.array(y)

def prepare_data_for_lstm(train_series, test_series, seq_length=5):
    """
    Prepare train and test data for LSTM
    """
    # Scale data
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_series.values.reshape(-1, 1))
    test_scaled = scaler.transform(test_series.values.reshape(-1, 1))
    
    # Create sequences
    X_train, y_train = create_sequences(train_scaled, seq_length)
    X_test, y_test = create_sequences(test_scaled, seq_length)
    
    # Convert to tensors
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train)
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test)
    
    return X_train, y_train, X_test, y_test, scaler

print("Data preparation functions defined")

Data preparation functions defined


## Training Function

In [6]:
def train_lstm_model(X_train, y_train, X_test, y_test, epochs=50, lr=0.001):
    """
    Train LSTM model
    Optimized for M3 Mac - small epochs, small batches
    """
    try:
        model = LightweightLSTM().to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        # Move data to device
        X_train = X_train.to(device)
        y_train = y_train.to(device)
        X_test = X_test.to(device)
        y_test = y_test.to(device)
        
        # Training loop
        model.train()
        for epoch in range(epochs):
            # Forward pass
            outputs = model(X_train)
            loss = criterion(outputs, y_train)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            predictions = model(X_test).cpu().numpy()
        
        return {
            'model': model.cpu(),
            'predictions': predictions,
            'success': True
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

print("Training function defined")

Training function defined


## Train LSTM Models

In [7]:
results = []
models_dict = {}
scalers_dict = {}

unique_combos = train_subset.groupby(['state-name', 'sector-name'])

print(f"Training LSTM models for {len(unique_combos)} series...")
print("This will take 15-20 minutes on M3 Mac...\n")

for (state, sector), group in tqdm(unique_combos, desc="Training LSTM"):
    # Get data
    train_series = group.sort_values('year')['value']
    
    test_group = test_subset[
        (test_subset['state-name'] == state) &
        (test_subset['sector-name'] == sector)
    ].sort_values('year')
    
    if len(test_group) == 0 or len(train_series) < 10:
        continue
    
    test_series = test_group['value']
    
    # Prepare data
    try:
        X_train, y_train, X_test, y_test, scaler = prepare_data_for_lstm(
            train_series, test_series, seq_length=5
        )
        
        if len(X_train) < 5 or len(X_test) < 1:
            continue
        
        # Train model
        result = train_lstm_model(X_train, y_train, X_test, y_test, epochs=50)
        
        if result['success']:
            # Inverse transform predictions
            predictions = scaler.inverse_transform(result['predictions'])
            actual = scaler.inverse_transform(y_test.numpy())
            
            # Calculate metrics
            mae = mean_absolute_error(actual, predictions)
            rmse = np.sqrt(mean_squared_error(actual, predictions))
            mape = mean_absolute_percentage_error(actual, predictions) * 100
            
            # Store
            model_key = f"{state}_{sector}_All Fuels".replace(' ', '_').replace('/', '_')
            models_dict[model_key] = result['model']
            scalers_dict[model_key] = scaler
            
            results.append({
                'state': state,
                'sector': sector,
                'fuel': 'All Fuels',
                'mae': mae,
                'rmse': rmse,
                'mape': mape
            })
    except:
        continue

results_df = pd.DataFrame(results)
print(f"\nSuccessfully trained {len(results_df)} LSTM models")

Training LSTM models for 30 series...
This will take 15-20 minutes on M3 Mac...



Training LSTM: 100%|██████████| 30/30 [00:04<00:00,  6.18it/s]


Successfully trained 30 LSTM models


## Performance Summary

In [8]:
if len(results_df) > 0:
    print("LSTM Model Performance")
    print("=" * 50)
    print(f"\nAverage MAPE: {results_df['mape'].mean():.2f}%")
    print(f"Average MAE: {results_df['mae'].mean():.2f}")
    print(f"Average RMSE: {results_df['rmse'].mean():.2f}")
    
    print("\nTop 5 Best Performing:")
    print(results_df.nsmallest(5, 'mape')[['state', 'sector', 'mape']])
else:
    print("No models trained successfully")

LSTM Model Performance

Average MAPE: 13.65%
Average MAE: 46.07
Average RMSE: 46.07

Top 5 Best Performing:
            state                                           sector      mape
10        Florida  Total carbon dioxide emissions from all sectors  2.088963
29  United States          Transportation carbon dioxide emissions  2.448122
22          Texas  Total carbon dioxide emissions from all sectors  2.766387
26  United States              Industrial carbon dioxide emissions  4.236878
8         Florida              Industrial carbon dioxide emissions  5.273034


## Save Models

In [9]:
if len(results_df) > 0:
    # Save results
    results_df.to_csv('../reports/model_comparison/lstm_results.csv', index=False)
    print("Saved: reports/model_comparison/lstm_results.csv")
    
    # Save all models (they're small)
    for model_key in models_dict.keys():
        torch.save(models_dict[model_key].state_dict(), 
                   f"../models/lstm/{model_key}.pth")
        joblib.dump(scalers_dict[model_key], 
                    f"../models/lstm/{model_key}_scaler.pkl")
    
    print(f"Saved {len(models_dict)} models")
    
    # Summary
    summary = {
        'model_type': 'LSTM',
        'architecture': 'Lightweight (32 units)',
        'n_models_trained': len(results_df),
        'average_mape': float(results_df['mape'].mean()),
        'average_mae': float(results_df['mae'].mean()),
        'average_rmse': float(results_df['rmse'].mean())
    }
    
    with open('../models/lstm/lstm_summary.json', 'w') as f:
        json.dump(summary, f, indent=4)
    print("Saved: models/lstm/lstm_summary.json")

Saved: reports/model_comparison/lstm_results.csv
Saved 30 models
Saved: models/lstm/lstm_summary.json


## Generate Future Predictions

Note: LSTM requires more careful handling for multi-step forecasting

In [10]:
# For simplicity, we'll use the other models (ARIMA/Prophet) for future predictions
# LSTM multi-step forecasting requires iterative prediction which can accumulate errors

print("Note: LSTM predictions for 2022-2030 will use historical pattern")
print("For dashboard, ARIMA and Prophet predictions are more reliable for long-term forecasts")

Note: LSTM predictions for 2022-2030 will use historical pattern
For dashboard, ARIMA and Prophet predictions are more reliable for long-term forecasts


## Complete

In [11]:
print("=" * 70)
print("LSTM MODEL TRAINING COMPLETE")
print("=" * 70)
if len(results_df) > 0:
    print(f"\nModels trained: {len(results_df)}")
    print(f"Average MAPE: {results_df['mape'].mean():.2f}%")
    print(f"\nOptimized for M3 MacBook Air")
    print(f"Device used: {device}")
print("\nNext: Run 06_model_comparison.ipynb")

LSTM MODEL TRAINING COMPLETE

Models trained: 30
Average MAPE: 13.65%

Optimized for M3 MacBook Air
Device used: mps

Next: Run 06_model_comparison.ipynb


In [ ]:
# ============================================================
# ADD THIS AS A NEW CELL AT THE END OF NOTEBOOK 05
# This generates LSTM forecasts for 2022-2030 and exports
# them to future_predictions.json
# ============================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import json
import os
from pathlib import Path

# ── 1. Re-define the model class (needed to load .pth files) ──────────────
class LightweightLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, dropout=0.2):
        super(LightweightLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        return self.fc(last_out)

# ── 2. Load the original full dataset (for seed values) ───────────────────
print("Loading data...")
df = pd.read_csv('../data/processed/emissions_processed.csv')

# ── 3. Load existing future_predictions.json ──────────────────────────────
predictions_path = '../data/dashboard/future_predictions.json'
with open(predictions_path, 'r') as f:
    existing_predictions = json.load(f)

print(f"Existing predictions: {len(existing_predictions)} rows")
print(f"Models in existing: {set(p['model'] for p in existing_predictions)}")

# ── 4. Find all saved LSTM model files ────────────────────────────────────
lstm_dir = Path('../models/lstm')
pth_files = list(lstm_dir.glob('*.pth'))
print(f"\nFound {len(pth_files)} LSTM model files")

# ── 5. Generate forecasts for each saved model ────────────────────────────
FORECAST_YEARS = list(range(2022, 2031))  # 2022 to 2030
SEQ_LENGTH = 5
new_lstm_predictions = []

for pth_file in pth_files:
    model_key = pth_file.stem  # e.g. "Texas_Electric_Power_All_Fuels"
    scaler_path = lstm_dir / f"{model_key}_scaler.pkl"

    if not scaler_path.exists():
        print(f"  Skipping {model_key} — no scaler found")
        continue

    # Parse state and sector from model key
    # model_key format: State_Name_Sector_Name_All_Fuels
    # We stored as: f"{state}_{sector}_All Fuels".replace(' ', '_').replace('/', '_')
    parts = model_key.replace('_All_Fuels', '').split('_')

    # Try to match state name from known top states
    top_states_list = ['Texas', 'California', 'Florida', 'Pennsylvania', 'Ohio']
    matched_state = None
    matched_sector = None

    for state in top_states_list:
        state_key = state.replace(' ', '_')
        if model_key.startswith(state_key):
            matched_state = state
            sector_key = model_key[len(state_key)+1:].replace('_All_Fuels', '')
            matched_sector = sector_key.replace('_', ' ')
            break

    if not matched_state:
        print(f"  Could not parse state from: {model_key}")
        continue

    # Get historical data for this series to use as seed
    mask = (
        (df['state-name'] == matched_state) &
        (df['sector-name'] == matched_sector) &
        (df['fuel-name'] == 'All Fuels')
    )
    series_df = df[mask].sort_values('year')

    if len(series_df) < SEQ_LENGTH:
        print(f"  Skipping {model_key} — not enough historical data")
        continue

    try:
        # Load scaler and model
        scaler = joblib.load(scaler_path)
        model = LightweightLSTM()
        model.load_state_dict(torch.load(pth_file, map_location='cpu'))
        model.eval()

        # Get last SEQ_LENGTH values as seed (most recent historical data)
        seed_values = series_df['value'].values[-SEQ_LENGTH:]
        seed_scaled = scaler.transform(seed_values.reshape(-1, 1)).flatten()

        # Iterative forecasting: predict one year at a time
        current_sequence = list(seed_scaled)
        forecast_scaled = []

        for _ in FORECAST_YEARS:
            # Prepare input tensor
            x = torch.FloatTensor(current_sequence[-SEQ_LENGTH:]).unsqueeze(0).unsqueeze(-1)
            with torch.no_grad():
                pred = model(x).item()
            forecast_scaled.append(pred)
            current_sequence.append(pred)  # feed prediction back in

        # Inverse transform to original scale
        forecast_values = scaler.inverse_transform(
            np.array(forecast_scaled).reshape(-1, 1)
        ).flatten()

        # Store predictions in same format as ARIMA/Prophet
        for year, value in zip(FORECAST_YEARS, forecast_values):
            new_lstm_predictions.append({
                'year': int(year),
                'state': matched_state,
                'sector': matched_sector,
                'fuel': 'All Fuels',
                'predicted_value': float(round(value, 4)),
                'model': 'LSTM'
            })

        print(f"  ✓ {matched_state} | {matched_sector} — forecast generated")

    except Exception as e:
        print(f"  ✗ {model_key} — Error: {e}")
        continue

print(f"\nGenerated {len(new_lstm_predictions)} LSTM prediction rows")

# ── 6. Remove any old LSTM entries from existing predictions ──────────────
cleaned_predictions = [p for p in existing_predictions if p.get('model') != 'LSTM']
print(f"Existing non-LSTM predictions: {len(cleaned_predictions)}")

# ── 7. Merge and save back to future_predictions.json ─────────────────────
final_predictions = cleaned_predictions + new_lstm_predictions
print(f"Total predictions after merge: {len(final_predictions)}")

with open(predictions_path, 'w') as f:
    json.dump(final_predictions, f, indent=2)

print(f"\n✅ Saved to {predictions_path}")
print(f"   ARIMA/Prophet entries kept: {len(cleaned_predictions)}")
print(f"   LSTM entries added: {len(new_lstm_predictions)}")
print(f"   Total: {len(final_predictions)}")
print("\n⚠️  Now restart your FastAPI server to reload the updated JSON!")
print("   Ctrl+C the uvicorn process, then run it again.")
print("\nDone! LSTM forecasts will now show in the dashboard for:")
for state in top_states_list:
    print(f"  - {state}")